In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA


## Load datasets

In [ ]:
path = '/content/Dataset'
arrays = []
data_list = []

for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:

            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])



In [ ]:
all_data = arrays[5]


## Balance datasets

In [ ]:
min_len = 2000

balanced_arrays = [arr for arr in arrays]

data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape


In [ ]:
def calculate_features(window):

    mean = np.mean(window)

    std_dev = np.std(window)

    max_val = np.max(window)

    min_val = np.min(window)

    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]


In [ ]:
def normalize_data(data):

    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    normalized_data = (data - mean) / std_dev

    return normalized_data


In [ ]:
pca = PCA(n_components=2)

def preprocessing(data):

    features = np.array([calculate_features(signal) for signal in data])

    # Historical shuffle has no fixed seed.
    np.random.shuffle(features)

    return features


## Training DBSCAN

In [ ]:
dataset = preprocessing(data)[:10000000]

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(dataset)
clustering_labels = kmeans.labels_


In [ ]:
import time
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
import matplotlib.pyplot as plt
from tqdm import tqdm

def measure_training_time_kmeans(dataset_size):
    dataset = preprocessing(data)[:dataset_size]

    start_time = time.time()
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(dataset)
    end_time = time.time()

    return end_time - start_time

def measure_training_time_dbscan(dataset_size):
    dataset = preprocessing(data)[:dataset_size]

    start_time = time.time()
    dbscan = DBSCAN()
    dbscan.fit(dataset)
    end_time = time.time()

    return end_time - start_time

dataset_sizes = range(10000, 100000, 10000)

training_times_kmeans = []
for size in tqdm(dataset_sizes, desc='Progress', unit='dataset'):
    elapsed_time = measure_training_time_kmeans(size)
    training_times_kmeans.append(elapsed_time)

training_times_dbscan = []
for size in tqdm(dataset_sizes, desc='Progress', unit='dataset'):
    elapsed_time = measure_training_time_dbscan(size)
    training_times_dbscan.append(elapsed_time)

plt.plot(dataset_sizes, training_times_kmeans, marker='o', label='KMeans')

plt.plot(dataset_sizes, training_times_dbscan, marker='o', label='DBSCAN')

plt.legend()

plt.title('Training Time vs. Dataset Size')
plt.xlabel('Dataset Size')
plt.ylabel('Training Time (s)')

plt.show()


In [ ]:
!cat /proc/cpuinfo


In [ ]:
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Number of clusters:", n_clusters)

n_anomalies = list(clustering_labels).count(-1)
print("Number of noise points:", n_anomalies)


In [ ]:
labelM = ["Normal","Maintenance","Compressor Fault"]

df = pd.DataFrame({'X': dataset[:, 0], 'Y': dataset[:, 1], 'Cluster': clustering_labels})

sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()


In [ ]:
label_names = ["Maintenance", "Compressor Fault","Normal" ]

df = pd.DataFrame({'Mean': dataset[:, 0], 'Std': dataset[:, 1], 'Cluster': clustering_labels})

df['Cluster'] = df['Cluster'].map({i: name for i, name in enumerate(label_names)})

sns.scatterplot(data=df, x='Mean', y='Std', hue='Cluster', palette='viridis', alpha=0.8)

plt.legend(title='Cluster')
plt.show()


## Inference

In [ ]:
def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):

    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino


In [ ]:
min_len = 2000

inference_data = all_data

inference_data = np.squeeze(inference_data,1)
inference_data.shape


In [ ]:
inference_data = preprocessing(inference_data)


In [ ]:
df2 = pd.DataFrame({'Mean': inference_data[:, 0], 'Std': inference_data[:, 4]})

sns.scatterplot(data=df2, x='Mean', y='Std', palette='viridis', alpha=0.5)
sns.scatterplot(data=df, x='Mean', y='Std', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()


In [ ]:
result = []
for elem in inference_data:
  result.append(distance(elem, dataset, clustering_labels))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(f"Nearest cluster for the new sample: {elemento_piu_ripetuto}")


In [ ]:
with open('kmeans_model.pkl', 'wb') as model_file:
    pickle.dump(kmeans, model_file)


In [ ]:
prova = inference_data[35]
prova = np.expand_dims(prova,0)
new_labels = kmeans.predict(prova)


In [ ]:
def compute_distances_from_clusters(kmeans_model, new_sample):
    """Return distances from a sample to the fitted cluster centroids."""

    distances = kmeans_model.transform(new_sample)

    return distances


In [ ]:
distances = compute_distances_from_clusters(kmeans, dataset)

plt.figure(figsize=(10, 6))
plt.hist(np.min(distances, axis=1), bins=30, edgecolor='black')
plt.title('Distribution of Distances from Cluster Centroids')
plt.xlabel('Minimum Distance')
plt.ylabel('Frequency')
plt.show()


In [ ]:
centroids = kmeans.cluster_centers_

distances_between_centroids = np.linalg.norm(centroids[0] - centroids[1])

print("Distance between centroids:", distances_between_centroids)


In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

silhouette_avg = silhouette_score(dataset, clustering_labels)
print(silhouette_avg)

ch_score = calinski_harabasz_score(dataset, clustering_labels)
print(ch_score)

db_score = davies_bouldin_score(dataset, clustering_labels)
print(db_score)
